In [ ]:
!pip install unsloth trl transformers accelerate peft aiohttp datasets


In [ ]:
HF_TOKEN = "FILL_IN_YOUR_TOKEN"
NGROK_URL = "FILL_IN_NGROK_URL"


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    token=HF_TOKEN
)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0,
    bias="none"
)


In [ ]:
import json

def build_ndrf_prompt(obs):
    my_obs = obs.get("ndrf_obs", {})
    return f"""You are the NDRF (National Disaster Response Force) Commander.\nCurrent Situation: Hour {obs.get('hours_elapsed')} of the disaster.\nTotal Trapped: {obs.get('total_trapped')}, Total Rescued: {obs.get('total_rescued')}\n\nYour Observations:\n{json.dumps(my_obs, indent=2)}\n\nYou CANNOT see the exact hospital capacity, government funds, NGO volunteer counts, or media sentiment.\n\nRespond ONLY with a JSON object containing:\n- \"ndrf_building_a_teams\": (int 0-3) number of teams to deploy to building A\n- \"ndrf_building_b_teams\": (int 0-3) number of teams to deploy to building B\n- \"ndrf_building_c_teams\": (int 0-3) number of teams to deploy to building C\n- \"ndrf_hospital_warning\": (bool) whether to issue a warning to hospitals\n- \"ndrf_casualty_estimate\": (int 100-400) estimated casualties\n"""

def build_hospital_prompt(obs):
    my_obs = obs.get("hospital_obs", {})
    return f"""You are the Hospital Administrator.\nCurrent Situation: Hour {obs.get('hours_elapsed')} of the disaster.\nTotal Trapped: {obs.get('total_trapped')}, Total Rescued: {obs.get('total_rescued')}\n\nYour Observations:\n{json.dumps(my_obs, indent=2)}\n\nYou CANNOT see the NDRF team deployments, government funds, NGO volunteer counts, or media sentiment.\n\nRespond ONLY with a JSON object containing:\n- \"hospital_surge_activated\": (bool) whether to activate surge capacity\n- \"hospital_ambulances_deployed\": (int 1-8) number of ambulances to dispatch\n"""

def build_govt_prompt(obs):
    my_obs = obs.get("govt_obs", {})
    return f"""You are the Government Emergency Coordinator.\nCurrent Situation: Hour {obs.get('hours_elapsed')} of the disaster.\nTotal Trapped: {obs.get('total_trapped')}, Total Rescued: {obs.get('total_rescued')}\n\nYour Observations:\n{json.dumps(my_obs, indent=2)}\n\nYou CANNOT see the specific NDRF team deployments, exact hospital capacity, NGO volunteer locations, or media sentiment.\n\nRespond ONLY with a JSON object containing:\n- \"emergency_declared\": (bool) whether to declare a full state of emergency\n- \"funds_released\": (int 0-2000) amount of funds to release\n"""

def build_ngo_prompt(obs):
    my_obs = obs.get("ngo_obs", {})
    return f"""You are the NGO Field Coordinator.\nCurrent Situation: Hour {obs.get('hours_elapsed')} of the disaster.\nTotal Trapped: {obs.get('total_trapped')}, Total Rescued: {obs.get('total_rescued')}\n\nYour Observations:\n{json.dumps(my_obs, indent=2)}\n\nYou CANNOT see the exact NDRF deployments, hospital capacity, government funds, or exact media sentiment.\n\nRespond ONLY with a JSON object containing:\n- \"ngo_building_assigned\": (string) location to send volunteers (choice of A, B, C, or shelter)\n- \"ngo_volunteers_deployed\": (int 50-200) number of volunteers deployed\n"""

def build_media_prompt(obs):
    my_obs = obs.get("media_obs", {})
    return f"""You are the Media Lead / Public Information Officer.\nCurrent Situation: Hour {obs.get('hours_elapsed')} of the disaster.\nTotal Trapped: {obs.get('total_trapped')}, Total Rescued: {obs.get('total_rescued')}\n\nYour Observations:\n{json.dumps(my_obs, indent=2)}\n\nYou CANNOT see the precise NDRF operations, hospital capacities, government funds, or exact NGO statistics.\n\nRespond ONLY with a JSON object containing:\n- \"press_briefing_issued\": (bool) whether to hold a live press briefing\n- \"panic_management_message\": (string) key message to broadcast (choice of Stay calm, Evacuate, or Help is coming)\n"""


In [ ]:
import aiohttp

async def env_reset(session):
    async with session.post(f"{NGROK_URL}/reset", json={}) as r:
        if r.status != 200:
            return {}
        data = await r.json()
        return data.get("observation", data)

async def env_step(session, action):
    async with session.post(f"{NGROK_URL}/step", json={"action": action}) as r:
        if r.status != 200:
            return {}, 0, True
        data = await r.json()
        obs = data.get("observation", {})
        reward = obs.get("current_reward", 0)
        done = obs.get("episode_done", False)
        return obs, reward, done


In [ ]:
async def collect_episode(model, tokenizer, session):
    obs = await env_reset(session)
    episode_data = []
    total_reward = 0
    
    for step in range(12):
        prompts = {
            "ndrf": build_ndrf_prompt(obs),
            "hospital": build_hospital_prompt(obs),
            "govt": build_govt_prompt(obs),
            "ngo": build_ngo_prompt(obs),
            "media": build_media_prompt(obs)
        }
        
        agent_responses = {}
        agent_raw_responses = {}
        
        for role, prompt in prompts.items():
            inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
            outputs = model.generate(**inputs, max_new_tokens=150, do_sample=True, temperature=0.7)
            response_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
            agent_raw_responses[role] = response_text
            
            try:
                start_idx = response_text.find('{')
                end_idx = response_text.rfind('}')
                if start_idx != -1 and end_idx != -1:
                    json_str = response_text[start_idx:end_idx+1]
                    agent_responses[role] = json.loads(json_str)
                else:
                    agent_responses[role] = {}
            except Exception:
                agent_responses[role] = {}
                
        action = {
            "ndrf_building_a_teams": agent_responses.get("ndrf", {}).get("ndrf_building_a_teams", 0),
            "ndrf_building_b_teams": agent_responses.get("ndrf", {}).get("ndrf_building_b_teams", 0),
            "ndrf_building_c_teams": agent_responses.get("ndrf", {}).get("ndrf_building_c_teams", 0),
            "ndrf_hospital_warning": agent_responses.get("ndrf", {}).get("ndrf_hospital_warning", False),
            "ndrf_casualty_estimate": agent_responses.get("ndrf", {}).get("ndrf_casualty_estimate", 100),
            "hospital_surge_activated": agent_responses.get("hospital", {}).get("hospital_surge_activated", False),
            "hospital_ambulances_deployed": agent_responses.get("hospital", {}).get("hospital_ambulances_deployed", 1),
            "emergency_declared": agent_responses.get("govt", {}).get("emergency_declared", False),
            "funds_released": agent_responses.get("govt", {}).get("funds_released", 0),
            "ngo_building_assigned": agent_responses.get("ngo", {}).get("ngo_building_assigned", "shelter"),
            "ngo_volunteers_deployed": agent_responses.get("ngo", {}).get("ngo_volunteers_deployed", 50),
            "press_briefing_issued": agent_responses.get("media", {}).get("press_briefing_issued", False),
            "panic_management_message": agent_responses.get("media", {}).get("panic_management_message", "Stay calm")
        }
        
        next_obs, reward, done = await env_step(session, action)
        
        step_data = {
            "prompts": prompts,
            "responses": agent_raw_responses,
            "reward": reward
        }
        episode_data.append(step_data)
        total_reward += reward
        obs = next_obs
        
        if done:
            break
            
    return episode_data, total_reward


In [ ]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW
import numpy as np
import matplotlib.pyplot as plt
import os
import asyncio

optimizer = AdamW(model.parameters(), lr=1e-5)
os.makedirs("/content/outputs", exist_ok=True)

episode_rewards = []
running_avg_rewards = []
losses = []

async def train_grpo():
    try:
        async with aiohttp.ClientSession() as session:
            for iteration in range(1, 101):
                episode_data, total_reward = await collect_episode(model, tokenizer, session)
                
                rewards = [step["reward"] for step in episode_data]
                if len(rewards) > 1:
                    r_mean = np.mean(rewards)
                    r_std = np.std(rewards)
                else:
                    r_mean = 0
                    r_std = 1
                
                loss_total = 0
                
                for step_idx, step in enumerate(episode_data):
                    norm_reward = (step["reward"] - r_mean) / (r_std + 1e-8)
                    
                    for role in step["prompts"]:
                        prompt = step["prompts"][role]
                        response = step["responses"][role]
                        
                        full_text = prompt + response
                        inputs = tokenizer(full_text, return_tensors="pt").to("cuda")
                        prompt_inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
                        prompt_length = prompt_inputs["input_ids"].shape[1]
                        
                        outputs = model(**inputs)
                        logits = outputs.logits[:, :-1, :]
                        labels = inputs["input_ids"][:, 1:]
                        
                        log_probs = F.log_softmax(logits, dim=-1)
                        token_log_probs = log_probs.gather(dim=-1, index=labels.unsqueeze(-1)).squeeze(-1)
                        
                        if token_log_probs.shape[1] > prompt_length - 1:
                            response_log_probs = token_log_probs[0, prompt_length-1:]
                            avg_log_prob = response_log_probs.mean()
                            
                            loss = -avg_log_prob * norm_reward
                            loss.backward()
                            loss_total += loss.item()
                
                optimizer.step()
                optimizer.zero_grad()
                
                episode_rewards.append(total_reward)
                running_avg = np.mean(episode_rewards[-10:]) if len(episode_rewards) > 0 else 0
                running_avg_rewards.append(running_avg)
                losses.append(loss_total)
                
                print(f"Iteration {iteration:3d} | Reward: {total_reward:6.1f} | Avg Reward: {running_avg:6.1f} | Loss: {loss_total:.4f}")
                
                if iteration % 25 == 0:
                    plt.figure(figsize=(10, 5))
                    plt.plot(episode_rewards, label='Episode Reward', alpha=0.3)
                    plt.plot(running_avg_rewards, label='Moving Avg (10)', color='red')
                    plt.title('Triage Wars GRPO Training Curve (Checkpoint)')
                    plt.xlabel('Training Episodes')
                    plt.ylabel('Total Reward')
                    plt.legend()
                    plt.savefig('/content/outputs/reward_curve_checkpoint.png')
                    plt.close()
                    
    except Exception as e:
        print(f"Training interrupted saving progress: {e}")

await train_grpo()


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(episode_rewards, label='Episode Reward', alpha=0.3)
plt.plot(running_avg_rewards, label='Moving Avg (10)', color='red')
plt.title('Triage Wars GRPO Training Curve')
plt.xlabel('Training Episodes')
plt.ylabel('Total Reward')
plt.legend()
plt.show()


In [ ]:
model.save_pretrained("/content/triage-wars-model")
tokenizer.save_pretrained("/content/triage-wars-model")
model.push_to_hub("132ragini/triage-wars-llm", token=HF_TOKEN)
tokenizer.push_to_hub("132ragini/triage-wars-llm", token=HF_TOKEN)

plt.figure(figsize=(10, 5))
plt.plot(episode_rewards, label='Episode Reward', alpha=0.3)
plt.plot(running_avg_rewards, label='Moving Avg (10)', color='red')
plt.title('Triage Wars GRPO Training Curve')
plt.xlabel('Training Episodes')
plt.ylabel('Total Reward')
plt.legend()
plt.savefig('/content/outputs/reward_curve_final.png')
plt.close()

print("Training complete and model saved.")
